> ⚠️ **Sandbox notice** — This notebook runs against the FiGuard shared demo environment (`sb_live_demo` key, `https://figuard-sandbox-g1ha.onrender.com`). It is **not for production use**. Budgets, events, and tokens may be wiped at any time without notice to keep the sandbox available for everyone. No uptime SLA. [Self-host for production →](https://github.com/figuard/figuard-core#self-hosting)

> 💡 **Quick start:** Select **Runtime → Run all** (Ctrl+F9) to run everything at once, then read the output below.

In [ ]:
# @title Step 1 — Install and configure FiGuard
import sys, importlib
!pip install "figuard>=0.3.0" --upgrade -q
# Flush any stale cached module from this runtime session
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} ready")
import uuid
USER_ID = f"demo_{uuid.uuid4().hex[:8]}"
print(f"Your session ID: {USER_ID}  (identifies your budgets in the dashboard)")

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient()

budget = client.create_budget(
    user_id=USER_ID,
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    authorization_expiry_seconds=300,
    intent_context="travel booking session",
    allocations=[
        {"category": "flight", "limit": 300.00, "enforcementMode": "CATEGORY_CONSTRAINED", "allowedCategories": ["flight"]},
        {"category": "hotel",  "limit": 200.00, "enforcementMode": "CATEGORY_CONSTRAINED", "allowedCategories": ["hotel"]},
    ],
)

print(f"✓ Budget created: {budget.id}")
print(f"  Total limit:   ${budget.total_limit}")
print(f"  Allocations:   $300 flights · $200 hotels")

session_token = budget.session_token

## Velocity controls — stopping runaway loops

`velocity_max_per_minute` caps the number of `authorize` calls in any rolling 60-second window. This catches retry loops before they accumulate meaningful spend — even if the budget is already exhausted and every call returns `BUDGET_EXHAUSTED`, the velocity counter still increments on each attempt.

Set it on `create_budget` alongside your other safety controls. It can also be updated later via `PATCH /budgets/{id}` without recreating the budget.

In [ ]:
budget_with_velocity = client.create_budget(
    user_id=USER_ID,
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    authorization_expiry_seconds=300,
    intent_context="travel booking session — velocity controls demo",
    allocations=[
        {"category": "flight", "limit": 300.00, "enforcementMode": "CATEGORY_CONSTRAINED", "allowedCategories": ["flight"]},
        {"category": "hotel",  "limit": 200.00, "enforcementMode": "CATEGORY_CONSTRAINED", "allowedCategories": ["hotel"]},
    ],
    velocity_max_per_minute=5,
)

print(f"✓ Budget created: {budget_with_velocity.id}")
print(f"  velocity_max_per_minute=5  →  6th call in any 60-second window is denied\n")

velocity_token = budget_with_velocity.primary_token.session_token

for i in range(1, 7):
    r = client.authorize(
        session_token=velocity_token,
        agent_id="retry_loop_agent",
        action_type="PURCHASE",
        description=f"Retry attempt {i}",
        requested_quantity=10.00,
        claimed_category="flight",
        idempotency_key=f"velocity-demo-{budget_with_velocity.id}-{i}",
    )
    label = r.decision if r.is_authorized else r.denial_reason
    print(f"  Call {i}: {label}")

print("\nCall 6 hit VELOCITY_LIMIT_EXCEEDED — runaway loop stopped before accumulating more spend.")
print(f"\n→ See the ledger: https://figuard-sandbox-g1ha.onrender.com/ui/budgets/{budget_with_velocity.id}")


In [ ]:
# Flight — within allocation
auth = client.authorize(
    session_token=session_token,
    agent_id="travel_agent",
    action_type="PURCHASE",
    description="JetBlue SFO→JFK roundtrip",
    requested_quantity=270.00,
    claimed_category="flight",
    idempotency_key="booking-001",
)
print(f"Flight booking:  {auth.decision} — ${auth.approved_quantity}")
client.confirm_event(auth.event_id, confirmed_quantity=267.00)
print("✓ Confirmed.")

# Hotel — within allocation
auth2 = client.authorize(
    session_token=session_token,
    agent_id="travel_agent",
    action_type="PURCHASE",
    description="Marriott 2 nights",
    requested_quantity=180.00,
    claimed_category="hotel",
    idempotency_key="booking-003",
)
print(f"Hotel booking:   {auth2.decision} — ${auth2.approved_quantity}")
client.confirm_event(auth2.event_id, confirmed_quantity=180.00)
print("✓ Confirmed.")

# Flight — exceeds remaining flight allocation ($300 - $267 = $33 left)
auth3 = client.authorize(
    session_token=session_token,
    agent_id="travel_agent",
    action_type="PURCHASE",
    description="Return flight upgrade",
    requested_quantity=150.00,
    claimed_category="flight",
    idempotency_key="booking-004",
)
print(f"Flight upgrade:  {auth3.decision} — {auth3.denial_reason}")
print("Nothing moved. Category limit enforced.")

print(f"\n→ Main budget:     https://figuard-sandbox-g1ha.onrender.com/ui/budgets/{budget.id}")
print(f"→ Velocity budget: https://figuard-sandbox-g1ha.onrender.com/ui/budgets/{budget_with_velocity.id}")